Use the torch.nn module to make the deep learning model easily

In [41]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [74]:
class Model(nn.Module):
    def __init__(self, num_features):
        # call the constructor function from the nn.Module
        super().__init__()
        
        # the only task of the nn.Lienar is to define a layer of m number of neurons that takes input from n nuerons
        # from previous layers, and perfomrm the linear combination wti the help of the orresponding weight
        self.layer = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()
        self.optimizer = torch.optim.SGD(self.layer.parameters(), lr=0.001)
    
    def forward(self, features):
        out = self.layer(features)
        self.y_pred = self.sigmoid(out).to(torch.float64)
        return self.y_pred
    
    def loss_function(self, real_class):
        loss_fn = nn.BCELoss()
        self.loss = loss_fn(self.y_pred, real_class)
        return self.loss.item()
    
    def optimization(self):
        # it is done so that there won't be previous gradient left which gets added to the new gradient
        self.optimizer.zero_grad()

        # calculate the gradient
        self.loss.backward()

        # backpropagation
        self.optimizer.step()
        return {'weights': self.layer.weight, 'bais' : self.layer.bias}
        

In [75]:
features = torch.rand(10,5)
print(features)
target = torch.randint(low=0, high= 2, size=(10,1), dtype=torch.float64)
print(target)

tensor([[0.0129, 0.3153, 0.5935, 0.2993, 0.3887],
        [0.2823, 0.1712, 0.2737, 0.0553, 0.9126],
        [0.4238, 0.0321, 0.9919, 0.4248, 0.3884],
        [0.3432, 0.0525, 0.4828, 0.1847, 0.7863],
        [0.9942, 0.8014, 0.6617, 0.0789, 0.6475],
        [0.8638, 0.6007, 0.0679, 0.6533, 0.2061],
        [0.1089, 0.8580, 0.4280, 0.1794, 0.1559],
        [0.8364, 0.4776, 0.7849, 0.9828, 0.9251],
        [0.2773, 0.2520, 0.2120, 0.4518, 0.8453],
        [0.4043, 0.2697, 0.9335, 0.6066, 0.7443]])
tensor([[1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.]], dtype=torch.float64)


In [76]:
ann = Model(features.shape[1])


In [78]:
y_pred = ann(features)
y_pred

tensor([[0.5448],
        [0.5500],
        [0.5636],
        [0.5588],
        [0.5368],
        [0.5851],
        [0.5162],
        [0.6454],
        [0.5888],
        [0.5958]], dtype=torch.float64, grad_fn=<ToCopyBackward0>)

In [79]:
ann.layer.weight

Parameter containing:
tensor([[ 0.0474, -0.0560, -0.0350,  0.4365,  0.1821]], requires_grad=True)

In [80]:
ann.layer.bias

Parameter containing:
tensor([0.0162], requires_grad=True)

In [81]:
loss = 1
for epoch in range(20):
    ann(features)
    loss = ann.loss_function(target)
    print("epoch: ", epoch, "===============>  loss:", loss)
    ann.optimization()

print('weight: ', ann.layer.weight)
print('bias: ', ann.layer.bias)

epoch:  0 ===============>  loss: 0.5901439107036783
epoch:  1 ===============>  loss: 0.5898983139331777
epoch:  2 ===============>  loss: 0.589652962342794
epoch:  3 ===============>  loss: 0.5894078965032132
epoch:  4 ===============>  loss: 0.5891630997933803
epoch:  5 ===============>  loss: 0.5889185350809104
epoch:  6 ===============>  loss: 0.5886742449185907
epoch:  7 ===============>  loss: 0.5884302675117965
epoch:  8 ===============>  loss: 0.5881864683381519
epoch:  9 ===============>  loss: 0.5879429924726973
epoch:  10 ===============>  loss: 0.5876997344027466
epoch:  11 ===============>  loss: 0.5874567511035219
epoch:  12 ===============>  loss: 0.587214036589944
epoch:  13 ===============>  loss: 0.5869715513231042
epoch:  14 ===============>  loss: 0.5867293520618706
epoch:  15 ===============>  loss: 0.586487420280961
epoch:  16 ===============>  loss: 0.5862457149173281
epoch:  17 ===============>  loss: 0.5860042778704115
epoch:  18 ===============>  loss: 0.5857

In [82]:
# ann.layer.parameters() ---> they are the generator so use iteration
for i in ann.layer.parameters():
    print(i)
print("================================================")
for i in ann.layer.named_parameters():
    print(i)

Parameter containing:
tensor([[ 0.0505, -0.0527, -0.0313,  0.4394,  0.1855]], requires_grad=True)
Parameter containing:
tensor([0.0228], requires_grad=True)
('weight', Parameter containing:
tensor([[ 0.0505, -0.0527, -0.0313,  0.4394,  0.1855]], requires_grad=True))
('bias', Parameter containing:
tensor([0.0228], requires_grad=True))


In [83]:
# importing the data as the dataframe from csv file
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [84]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [85]:
# rempving the unwanted features like id and unnamed: 32
df.drop(columns=['id', "Unnamed: 32"], inplace = True)

In [86]:
df.shape

(569, 31)

In [87]:
features = df.iloc[:,1:]
target = df.iloc[:,0]

In [88]:
# split the data set into train and test
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=43)

In [89]:
x_train

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
415,11.89,21.17,76.39,433.8,0.09773,0.08120,0.025550,0.021790,0.2019,0.06290,...,13.05,27.21,85.09,522.9,0.1426,0.21870,0.116400,0.08263,0.3075,0.07351
256,19.55,28.77,133.60,1207.0,0.09260,0.20630,0.178400,0.114400,0.1893,0.06232,...,25.05,36.27,178.60,1926.0,0.1281,0.53290,0.425100,0.19410,0.2818,0.10050
420,11.57,19.04,74.20,409.7,0.08546,0.07722,0.054850,0.014280,0.2031,0.06267,...,13.07,26.98,86.43,520.5,0.1249,0.19370,0.256000,0.06664,0.3035,0.08284
448,14.53,19.34,94.25,659.7,0.08388,0.07800,0.088170,0.029250,0.1473,0.05746,...,16.30,28.39,108.10,830.5,0.1089,0.26490,0.377900,0.09594,0.2471,0.07463
195,12.91,16.33,82.53,516.4,0.07941,0.05366,0.038730,0.023770,0.1829,0.05667,...,13.88,22.00,90.81,600.6,0.1097,0.15060,0.176400,0.08235,0.3024,0.06949
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16,14.68,20.13,94.74,684.5,0.09867,0.07200,0.073950,0.052590,0.1586,0.05922,...,19.07,30.88,123.40,1138.0,0.1464,0.18710,0.291400,0.16090,0.3029,0.08216
58,13.05,19.31,82.61,527.2,0.08060,0.03789,0.000692,0.004167,0.1819,0.05501,...,14.23,22.25,90.24,624.1,0.1021,0.06191,0.001845,0.01111,0.2439,0.06289
277,18.81,19.98,120.90,1102.0,0.08923,0.05884,0.080200,0.058430,0.1550,0.04996,...,19.96,24.30,129.00,1236.0,0.1243,0.11600,0.221000,0.12940,0.2567,0.05737
255,13.96,17.05,91.43,602.4,0.10960,0.12790,0.097890,0.052460,0.1908,0.06130,...,16.39,22.07,108.10,826.0,0.1512,0.32620,0.320900,0.13740,0.3068,0.07957


In [90]:
# StandardScaling
scaler = StandardScaler()
# fit_transform will first learn the mean and the s.d. of the trainning data and then do the calculation
# trasnform will that the mean and s.d.of the training data and work for the testing data
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [91]:
x_train_scaled[0]

array([-0.63904656,  0.43769321, -0.64504029, -0.62945156,  0.09920259,
       -0.44019265, -0.79716811, -0.6983249 ,  0.72114937,  0.01190875,
       -0.46133074,  0.00649877, -0.45347087, -0.44690619,  0.96005806,
        0.29047614, -0.61987796, -0.40248677,  0.27735408, -0.64126786,
       -0.67242081,  0.24448954, -0.66651354, -0.63112959,  0.44623997,
       -0.24509593, -0.76226753, -0.49515846,  0.26325243, -0.58507738])

In [92]:
# LabelEncoder
y_train
encoder = LabelEncoder()
y_train_labeled = encoder.fit_transform(y_train)
y_test_labeled = encoder.transform(y_test)

In [93]:
# as we work on pyTorch, tensor is required for computation
features_train = torch.from_numpy(x_train_scaled).to(torch.float32)
features_test = torch.from_numpy(x_test_scaled).to(torch.float32)
target_train = torch.from_numpy(y_train_labeled).to(torch.float32)
target_test = torch.from_numpy(y_test_labeled).to(torch.float32)

In [95]:

# making a multi-layer ANN with the overfitting control method dropout and fast convergene method BatchNormalization

class ANN_Model(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        # ANN model:
        self.model = nn.Sequential(
            nn.Linear(num_features,3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )
        # optimizer function: 
        self.optim = torch.optim.SGD(self.model.parameters(), lr = 0.1)

    def forward(self, features):
        self.y_pred = self.model(features)
        return self.y_pred
    
    def loss_function(self, y_real):
        loss_fn = nn.BCELoss()
        self.loss = loss_fn(self.y_pred, y_real)
        return self.loss.item()
    
    def optimizer(self):
        self.optim.zero_grad()
        self.loss.backward()
        self.optim.step()
        

In [110]:
model = ANN_Model(features_train.shape[1])

In [111]:
for i in model.model.named_parameters():
    print(i)

('0.weight', Parameter containing:
tensor([[-0.0869,  0.1342,  0.1077, -0.0710, -0.0345,  0.0998,  0.0057,  0.1549,
         -0.0353, -0.0169,  0.0344,  0.1253,  0.1791,  0.1125, -0.1048,  0.1802,
          0.0917,  0.0879, -0.0609, -0.0270, -0.0222,  0.1555,  0.1557, -0.1556,
          0.0405, -0.0481, -0.0454,  0.0906,  0.1062,  0.0632],
        [-0.0792,  0.0155,  0.0764, -0.1798,  0.1012,  0.1711,  0.1539,  0.1806,
         -0.1334,  0.1554, -0.0623,  0.0016,  0.0387, -0.0338,  0.1513,  0.0757,
          0.0176, -0.0942,  0.1384, -0.1508,  0.1016,  0.1576, -0.0112,  0.1687,
          0.1436, -0.1160, -0.0843, -0.0797,  0.1743, -0.0712],
        [-0.0985, -0.0605,  0.0628, -0.0206, -0.0532,  0.1382, -0.1055, -0.1463,
          0.0717,  0.1216, -0.0777,  0.1523, -0.0062, -0.0421,  0.1776, -0.1505,
          0.1150,  0.0709, -0.1405,  0.1390,  0.0933, -0.1227, -0.0541, -0.1602,
          0.1728,  0.0749, -0.0815, -0.1238,  0.1728,  0.1060]],
       requires_grad=True))
('0.bias', Para

In [112]:
epoch = 20

In [113]:
# training for 20 epochs
import time
for i in range(30):
    model.forward(features_train)
    loss = model.loss_function(target_train.reshape(features_train.shape[0],1))
    model.optimizer()
    print("epoch:",i, "loss:", loss)

epoch: 0 loss: 0.7347590923309326
epoch: 1 loss: 0.6971429586410522
epoch: 2 loss: 0.6658798456192017
epoch: 3 loss: 0.6403952836990356
epoch: 4 loss: 0.6185813546180725
epoch: 5 loss: 0.5999800562858582
epoch: 6 loss: 0.5834434032440186
epoch: 7 loss: 0.568396806716919
epoch: 8 loss: 0.5547086000442505
epoch: 9 loss: 0.5419517755508423
epoch: 10 loss: 0.5299066305160522
epoch: 11 loss: 0.5185463428497314
epoch: 12 loss: 0.5076645612716675
epoch: 13 loss: 0.4970163106918335
epoch: 14 loss: 0.48665839433670044
epoch: 15 loss: 0.47666388750076294
epoch: 16 loss: 0.46700015664100647
epoch: 17 loss: 0.4575692415237427
epoch: 18 loss: 0.44828951358795166
epoch: 19 loss: 0.43912726640701294
epoch: 20 loss: 0.4300323724746704
epoch: 21 loss: 0.4209619164466858
epoch: 22 loss: 0.4119032323360443
epoch: 23 loss: 0.40284281969070435
epoch: 24 loss: 0.39368516206741333
epoch: 25 loss: 0.3844834268093109
epoch: 26 loss: 0.3751619756221771
epoch: 27 loss: 0.3659387528896332
epoch: 28 loss: 0.356780

In [116]:
for i in model.model.parameters():
    print(i)

Parameter containing:
tensor([[-0.2217,  0.0847, -0.0283, -0.1964, -0.1077,  0.0056, -0.1094,  0.0218,
         -0.0891,  0.0038, -0.0584,  0.1575,  0.0886,  0.0223, -0.0834,  0.1441,
          0.0515,  0.0192, -0.0470, -0.0264, -0.1616,  0.0975,  0.0154, -0.2817,
         -0.0435, -0.1470, -0.1615, -0.0486,  0.0272,  0.0128],
        [ 0.0925,  0.1245,  0.2468, -0.0193,  0.1307,  0.2414,  0.2821,  0.3309,
         -0.0965,  0.0704,  0.0385, -0.0188,  0.1319,  0.0663,  0.0931,  0.0970,
          0.0629, -0.0216,  0.0997, -0.1782,  0.2825,  0.2795,  0.1676,  0.3328,
          0.2090, -0.0175,  0.0540,  0.0924,  0.2562, -0.0390],
        [-0.0104, -0.0031,  0.1510,  0.0570, -0.0466,  0.1792, -0.0375, -0.0691,
          0.0851,  0.0691, -0.0354,  0.1327,  0.0347,  0.0029,  0.1109, -0.1514,
          0.1192,  0.0905, -0.1648,  0.0935,  0.1855, -0.0530,  0.0384, -0.0810,
          0.1958,  0.1421, -0.0030, -0.0311,  0.2312,  0.1277]],
       requires_grad=True)
Parameter containing:
tensor(

In [117]:
# Testing
y = model(features_test)
model.loss_function(target_test.reshape(features_test.shape[0],1))

0.3546098470687866

Now we can u se the droput layer and also apply the BatchNormalization in each output of the neuron in each of the hidden layer

Dropout layer is used to dropout or remove/disable certain number of neuron so that the neurons can learn the general pattern through every feature/e=neruon